In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [2]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(X.shape)
print(y.value_counts())


(569, 30)
target
1    357
0    212
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [5]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier()
}


In [6]:
results = []

for name, model in models.items():
    
    # choose scaled or unscaled
    if name in ["Logistic Regression", "SVM", "KNN"]:
        Xtr, Xte = X_train_scaled, X_test_scaled
    else:
        Xtr, Xte = X_train, X_test
    
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    
    probs = model.predict_proba(Xte)[:,1]
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    results.append([name, acc, f1, auc])


In [7]:
results

[['Logistic Regression',
  0.9824561403508771,
  0.9861111111111112,
  0.9953703703703703],
 ['Decision Tree', 0.9122807017543859, 0.9295774647887324, 0.9107142857142856],
 ['Random Forest', 0.956140350877193, 0.9655172413793104, 0.994212962962963],
 ['SVM', 0.9824561403508771, 0.9861111111111112, 0.9950396825396826],
 ['KNN', 0.956140350877193, 0.9655172413793104, 0.9788359788359788]]

In [8]:
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1", "ROC-AUC"])
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

print(results_df)


                 Model  Accuracy        F1   ROC-AUC
0  Logistic Regression  0.982456  0.986111  0.995370
3                  SVM  0.982456  0.986111  0.995040
2        Random Forest  0.956140  0.965517  0.994213
4                  KNN  0.956140  0.965517  0.978836
1        Decision Tree  0.912281  0.929577  0.910714


In [9]:
best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)


Best model: Logistic Regression


In [10]:
best_model = models[best_model_name]

if best_model_name in ["Logistic Regression", "SVM", "KNN"]:
    Xtr, Xte = X_train_scaled, X_test_scaled
else:
    Xtr, Xte = X_train, X_test

best_model.fit(Xtr, y_train)
preds = best_model.predict(Xte)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

